# Integração do dataset e bibliotecas

In [ ]:
!pip install anonymizedf

import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import hashlib
import os

path = kagglehub.dataset_download("preritsaxena/fraud-detection")

print("Dataset baixado em:", path)

arquivos = os.listdir(path)
print("Arquivos encontrados:", arquivos)

csv_path = os.path.join(path, arquivos[0])

df = pd.read_csv(csv_path)

print("\nFirst 5 records:")
display(df.head())

Using Colab cache for faster access to the 'fraud-detection' dataset.
Dataset baixado em: /kaggle/input/fraud-detection
Arquivos encontrados: ['Fraud_Data.csv']

First 5 records:


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,is_fraud
0,22058,2015-02-24 22:55:49,2015-04-18 2:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 1:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 7:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


# Verificar dados duplicados

In [ ]:
duplicados = df.duplicated(["user_id", "signup_time", "purchase_time", "purchase_value", "device_id", "source", "browser", "sex", "age", "ip_address", "is_fraud"]).sum()
display("A quantidade de dados duplicados é: ", duplicados)

'A quantidade de dados duplicados é: '

np.int64(0)

# Verificar users e devices duplicados

In [ ]:
#Esta verificação é importante pois podem haver bots fazendo compras ao mesmo tempo usando o mesmo id de usuario e dispositivo

user_duplicado = df.duplicated(subset=['user_id']).sum()
print(f"Quantidade de usuarios duplicados: {user_duplicado}")

compras_suspeitas = df.duplicated(subset=['purchase_time','device_id']).sum()
print(f"Quantidadede de compras suspeitas: {compras_suspeitas}")


Quantidade de usuarios duplicados: 0
Quantidadede de compras suspeitas: 0


# Formatação de dados.

Verificar dados irrelevantes.

In [ ]:
filtro = (
    df['device_id'].str.contains(r'[^a-zA-Z0-9]', na=False) |
    df['source'].str.contains(r'[^a-zA-Z0-9]', na=False) |
    df['browser'].str.contains(r'[^a-zA-Z0-9]', na=False)|
    df['sex'].str.contains(r'[^a-zA-Z0-9]', na=False)
)
print(f"Quantidade de dados irrelevantes: {filtro}")
if filtro.sum() > 0:
    display(df[filtro].head())

Quantidade de dados irrelevantes: 0         False
1         False
2         False
3         False
4         False
          ...  
151107    False
151108    False
151109    False
151110    False
151111    False
Length: 151112, dtype: bool


Remover dados irrelevantes

In [ ]:
apagar = (
    df['device_id'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['source'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['browser'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['sex'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['age'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['user_id'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['signup_time'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['purchase_time'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['purchase_value'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['ip_address'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['is_fraud'].replace(r'[^a-zA-Z0-9]','',regex=True )
)

print("Colunas limpas")

Colunas limpas


# Ocultar dados sensiveis

In [ ]:

# Ocultar dados sensíveis

# Anonimização da coluna 'sex': (mantido conforme solicitado)
df['sex'] = pd.factorize(df['sex'])[0] # Feminino = 1, Masculino = 0

# Anonimização da coluna 'age': Multiplicando a idade por 2
# Garantimos que a coluna 'age' seja numérica. Valores não numéricos (como 'Unknown' de execuções anteriores)
# serão convertidos para NaN e depois preenchidos com 0 antes da multiplicação, garantindo a operação.
df['age'] = pd.to_numeric(df['age'], errors='coerce').fillna(0).astype(int) * 2  # Idade multiplicada por 2 para ocultação

# Anonimização das colunas 'ip_address' e 'device_id': Utilizando um padrão de hash simplificado (primeiros 8 caracteres SHA-256)
# Esta técnica transforma os valores originais em um hash encurtado, tornando-o mais simples visualmente.
# Ainda permite rastrear se o mesmo IP ou Device ID original aparece novamente, mas com um padrão mais conciso.

def hash_value(value):
    if pd.isna(value):
        return None
    # Retorna os primeiros 8 caracteres do hash SHA-256 para um padrão mais simples
    return hashlib.sha256(str(value).encode()).hexdigest()[:8]

df['ip_address'] = df['ip_address'].apply(hash_value) # IP Address ocultado com hash SHA-256 (primeiros 8 caracteres)
df['device_id'] = df['device_id'].apply(hash_value)   # Device ID ocultado com hash SHA-256 (primeiros 8 caracteres)

print("Tabela com dados anonimizados:")
df.head()


Tabela com dados anonimizados conforme as novas regras (hash simplificado):


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,is_fraud
0,22058,2015-02-24 22:55:49,2015-04-18 2:47:11,34,4007879c,SEO,Chrome,0,156,db63c069,0
1,333320,2015-06-07 20:39:50,2015-06-08 1:38:54,16,364922ed,Ads,Chrome,1,212,630fc5c5,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,03f804ea,SEO,Opera,0,212,9291c2e1,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,a4876f4e,SEO,Safari,0,164,c751e492,0
4,221365,2015-07-21 7:09:52,2015-09-09 18:40:53,39,a428d9bf,Ads,Safari,0,180,b994cafd,0


## cálculos para extrair características dos dados e entender a distribuição deles.

Calculado media, mediaa e a moda

In [ ]:
print("Medidas de Localidade para 'purchase_value':")
mean_purchase_value = df['purchase_value'].mean()
median_purchase_value = df['purchase_value'].median()
mode_purchase_value = df['purchase_value'].mode()

print(f"  Média (purchase_value): {mean_purchase_value:.2f}")
print(f"  Mediana (purchase_value): {median_purchase_value:.2f}")
print(f"  Moda (purchase_value): {mode_purchase_value.tolist()}") # tolist() para exibir todos os modos se houver mais de um

print("\nMedidas de Localidade para 'age':")
mean_age = df['age'].mean()
median_age = df['age'].median()
mode_age = df['age'].mode()

print(f"  Média (age): {mean_age:.2f}")
print(f"  Mediana (age): {median_age:.2f}")
print(f"  Moda (age): {mode_age.tolist()}") # tolist() para exibir todos os modos se houver mais de um


Medidas de Localidade para 'purchase_value':
  Média (purchase_value): 36.94
  Mediana (purchase_value): 35.00
  Moda (purchase_value): [28]

Medidas de Localidade para 'age':
  Média (age): 132.56
  Mediana (age): 132.00
  Moda (age): [128]


Calculando medidas de espalhamento "Amplitude,Variante, Desvio Padrão"



In [ ]:
print("\nMedidas de Espalhamento para 'purchase_value':")
range_purchase_value = df['purchase_value'].max() - df['purchase_value'].min()
variance_purchase_value = df['purchase_value'].var()
std_dev_purchase_value = df['purchase_value'].std()

print(f"  Amplitude (purchase_value): {range_purchase_value:.2f}")
print(f"  Variância (purchase_value): {variance_purchase_value:.2f}")
print(f"  Desvio-Padrão (purchase_value): {std_dev_purchase_value:.2f}")

print("\nMedidas de Espalhamento para 'age':")
range_age = df['age'].max() - df['age'].min()
variance_age = df['age'].var()
std_dev_age = df['age'].std()

print(f"  Amplitude (age): {range_age:.2f}")
print(f"  Variância (age): {variance_age:.2f}")
print(f"  Desvio-Padrão (age): {std_dev_age:.2f}")



Medidas de Espalhamento para 'purchase_value':
  Amplitude (purchase_value): 145.00
  Variância (purchase_value): 335.72
  Desvio-Padrão (purchase_value): 18.32

Medidas de Espalhamento para 'age':
  Amplitude (age): 232.00
  Variância (age): 1188.25
  Desvio-Padrão (age): 34.47
